
# SyntaxError

Example script with invalid Python syntax



In [ ]:
"""
Dust attenuation: uncertainty in SED from dust parameter estimation
==================================================================

Demonstrates dust attenuation effects and how fitting uncertainty propagates
to the recovered SED. A galaxy with free dust parameters (tau_bc and tau_diff)
is fit with MAP, showing the best-fit SED plus mock perturbation envelopes
to illustrate the uncertainty range from photometric noise.

Reference: Calzetti et al. 2000, ApJ, 533, 682 (attenuation law);
Conroy 2013, ARA&A, 51, 393 (SED fitting uncertainties).
"""

import os

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"  # suppress XLA/PjRt C++ INFO+WARNING logs

import warnings

import jax
import matplotlib.pyplot as plt
import numpy as np

import tengri
from tengri.analysis.plotting import setup_style

setup_style()
warnings.filterwarnings("ignore", message=".*BakedInBackend.*")

jax.config.update("jax_enable_x64", True)

ssp = tengri.load_ssp()
bands = ["sdss_u", "sdss_g", "sdss_r", "sdss_i", "sdss_z"]
obs = tengri.Observation(photometry=tengri.Photometry.from_names(bands))

# Model with free dust parameters
model = tengri.SEDModel.build(
    ssp,
    observation=obs,
    sfh={
        "type": "tsnorm",
        "log_total_mass": 10.0, 2.5),
        "peak_lbt_gyr": tengri.Uniform(0.5, 12.0),
        "width_gyr": tengri.Uniform(0.3, 5.0),
        "skew": tengri.Uniform(-1.0, 1.5),
        "trunc": tengri.Uniform(1.0, 10.0),
        "logzsol": tengri.Uniform(-2.0, 0.2),
    },
    dust={
        "type": "two_component",
        "tau_bc": tengri.Uniform(0.0, 2.0),
        "tau_diff": tengri.Uniform(0.0, 1.5),
        "slope": tengri.Fixed(-0.7),
    },
    redshift=tengri.Fixed(0.1),
)

# Generate mock data
key = jax.random.PRNGKey(42)
truth_params = {
    "sfh_tsnorm_log_total_mass": 0.8,
    "sfh_tsnorm_peak_lbt_gyr": 3.0,
    "sfh_tsnorm_width_gyr": 1.5,
    "sfh_tsnorm_skew": 0.3,
    "sfh_tsnorm_trunc": 5.0,
    "met_logzsol": -0.1,
    "dust_tau_bc": 0.5,
    "dust_tau_diff": 0.3,
    "dust_slope": -0.7,
    "redshift": 0.1,
}
mock = model.mock(truth_params, snr=20.0, key=key)

# Fit with MAP
forward = tengri.ForwardModel.build(sed=model, observation=obs)
posterior = forward.fit(
    mock.flux_obs,
    mock.noise,
    method="map",
    optimizer="adam",
    n_steps=300,
    verbose=False,
)

# Generate uncertainty envelope via parameter perturbation
n_resample = 100
key_pert = jax.random.PRNGKey(999)
posterior_photometry = []

for _i in range(n_resample):
    key_pert, subkey = jax.random.split(key_pert)
    perturbation = 0.15 * jax.random.normal(subkey, shape=(len(posterior.params),))
    param_names = list(posterior.params.keys())
    perturbed = {
        name: float(posterior.params[name]) + 0.1 * pert
        for name, pert in zip(param_names, perturbation)
    }
    phot_i = model.predict_photometry(perturbed)
    posterior_photometry.append(np.array(phot_i))

posterior_photometry = np.array(posterior_photometry)

# Compute envelopes
phot_map = np.asarray(model.predict_photometry(posterior.params))
phot_p16 = np.percentile(posterior_photometry, 16, axis=0)
phot_p84 = np.percentile(posterior_photometry, 84, axis=0)
phot_p2_5 = np.percentile(posterior_photometry, 2.5, axis=0)
phot_p97_5 = np.percentile(posterior_photometry, 97.5, axis=0)

# Plot
fig, ax = plt.subplots(figsize=(9, 5))

wave_eff = np.array([3551, 4686, 6166, 7480, 8932])
band_labels = ["u", "g", "r", "i", "z"]

ax.fill_between(
    wave_eff,
    phot_p2_5,
    phot_p97_5,
    color="C0",
    alpha=0.2,
    label="2σ credible region",
)
ax.fill_between(
    wave_eff,
    phot_p16,
    phot_p84,
    color="C0",
    alpha=0.4,
    label="1σ credible region",
)

ax.plot(wave_eff, phot_map, "C0-", lw=2.5, label="MAP fit", marker="o", ms=8)

ax.errorbar(
    wave_eff,
    np.array(mock.flux_obs),
    yerr=np.array(mock.noise),
    fmt="o",
    color="k",
    ms=7,
    capsize=3,
    label="Observed",
    zorder=5,
)

ax.scatter(
    wave_eff,
    np.array(mock.flux_true),
    marker="s",
    s=80,
    facecolors="none",
    edgecolors="red",
    lw=1.5,
    label="Truth",
    zorder=4,
)

ax.set_xlabel(r"Wavelength [$\mathrm{\AA}$]")
ax.set_ylabel(r"$f_\nu$  [erg s$^{-1}$ cm$^{-2}$ Hz$^{-1}$]")
ax.legend(frameon=False, loc="upper right")
ax.set_xticks(wave_eff)
ax.set_xticklabels(band_labels)

fig.tight_layout()
plt.savefig("plot_workflow_dust_mc_resampling.png", dpi=150, bbox_inches="tight")